In [3]:
import warnings
warnings.filterwarnings("ignore")

In [6]:
# Connect to the running Spark session
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

In [7]:
# Get the existing session (or create new one)
try:
    spark = SparkSession.getActiveSession()
    if not spark:
        spark = SparkSession.builder.getOrCreate()
except:
    spark = SparkSession.builder.master("local[*]").getOrCreate()

print(f"✅ Connected to Spark {spark.version}")

✅ Connected to Spark 4.1.2


In [8]:
spark

In [ ]:
import builtins  # Add this at the top
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType, DecimalType
# DO NOT import round from PySpark
from pyspark.sql.functions import rand, randn, when, col, lit, abs, current_date, date_sub, expr
import random
from datetime import datetime, timedelta
import pandas as pd
import numpy as np
import os

# Use Python's built-in round
py_round = builtins.round

# ============ 1. CREATE PRODUCT TABLE ============

print("📦 Creating Product Table with 5000+ products...")

# Product categories with sugar-related focus
categories = {
    'Sugar_Products': ['White Sugar', 'Brown Sugar', 'Powdered Sugar', 'Cane Sugar', 'Beet Sugar', 
                       'Organic Sugar', 'Raw Sugar', 'Demerara Sugar', 'Muscovado Sugar', 'Turbinado Sugar',
                       'Sugar Cubes', 'Sugar Syrup', 'Maple Sugar', 'Coconut Sugar', 'Palm Sugar',
                       'Sugarcane Juice', 'Sugar Cane Sticks', 'Jaggery', 'Rock Sugar', 'Pearl Sugar'],
    
    'Dairy': ['Milk', 'Cheese', 'Yogurt', 'Butter', 'Cream', 'Ice Cream', 'Condensed Milk', 
              'Evaporated Milk', 'Sour Cream', 'Cottage Cheese'],
    
    'Beverages': ['Cola', 'Lemonade', 'Fruit Juice', 'Sports Drink', 'Energy Drink', 'Soda', 
                  'Iced Tea', 'Flavored Water', 'Coffee', 'Tea'],
    
    'Bakery': ['Bread', 'Cake', 'Pastry', 'Cookies', 'Donuts', 'Muffins', 'Brownies', 'Croissants',
               'Baguette', 'Rye Bread'],
    
    'Fruits': ['Apple', 'Banana', 'Orange', 'Grapes', 'Strawberries', 'Watermelon', 'Peach', 'Pear',
               'Pineapple', 'Mango', 'Kiwi', 'Blueberries'],
    
    'Vegetables': ['Potato', 'Tomato', 'Onion', 'Carrot', 'Broccoli', 'Spinach', 'Lettuce', 'Cucumber',
                   'Pepper', 'Garlic', 'Ginger', 'Mushroom'],
    
    'Snacks': ['Chips', 'Crackers', 'Candy', 'Chocolate', 'Gum', 'Mints', 'Popcorn', 'Pretzels',
               'Nut Mix', 'Granola Bars', 'Fruit Snacks', 'Pudding'],
    
    'Grains': ['Rice', 'Wheat', 'Oats', 'Barley', 'Quinoa', 'Cornmeal', 'Flour', 'Cereal',
               'Pasta', 'Noodles', 'Bread Crumbs', 'Couscous'],
    
    'Condiments': ['Ketchup', 'Mustard', 'Mayonnaise', 'Soy Sauce', 'Vinegar', 'Oil', 'Salt',
                   'Pepper', 'Spices', 'Herbs', 'Honey', 'Syrup'],
    
    'Meat': ['Chicken', 'Beef', 'Pork', 'Fish', 'Shrimp', 'Turkey', 'Lamb', 'Sausage', 'Bacon', 'Ham']
}

# Generate product data
products = []
product_id = 1

for category, items in categories.items():
    # Make sugar products more prominent
    sugar_boost = 5 if category == 'Sugar_Products' else 1
    
    for item in items:
        # Generate 2-5 variants per product
        for variant in range(sugar_boost):
            # ✅ Use py_round (Python's built-in round)
            base_price_multiplier = random.uniform(0.85, 1.15)
            base_price = py_round(random.uniform(1.99, 45.99) * base_price_multiplier, 2)
            # Ensure minimum price
            if base_price < 0.99:
                base_price = 0.99
            
            # Create sugar-specific pricing
            is_sugar_related = category == 'Sugar_Products' or 'Sugar' in item
            if is_sugar_related:
                price = py_round(base_price * (1 + random.uniform(-0.3, 0.5)), 2)
            else:
                price = py_round(base_price, 2)
            
            # Ensure price is never negative or zero
            if price < 0.99:
                price = 0.99
            
            # Sugar content indicator
            sugar_content = random.randint(0, 100)
            if is_sugar_related:
                sugar_content = random.randint(60, 100)
            elif category in ['Beverages', 'Bakery', 'Snacks']:
                sugar_content = random.randint(20, 80)
            else:
                sugar_content = random.randint(0, 30)
            
            product = {
                'product_id': product_id,
                'product_name': f"{item} - {chr(65 + variant)}" if variant > 0 else item,
                'category': category,
                'sub_category': item,
                'price': price,
                'cost': py_round(price * random.uniform(0.4, 0.8), 2),
                'sugar_content_grams': sugar_content,
                'is_sugar_product': is_sugar_related,
                'weight_kg': py_round(random.uniform(0.1, 5.0), 2),
                'brand': random.choice(['SweetLife', 'NatureFresh', 'PureGood', 'DailyDeli', 
                                        'FarmHouse', 'GoldenHarvest', 'SweetCane', 'SugarKing']),
                'supplier': random.choice(['Supplier A', 'Supplier B', 'Supplier C', 'Supplier D', 
                                           'Supplier E', 'Supplier F']),
                'stock_quantity': random.randint(0, 1000),
                'reorder_level': random.randint(5, 100),
                'expiry_days': random.randint(7, 365)
            }
            products.append(product)
            product_id += 1

# Create product DataFrame
product_df = spark.createDataFrame(products)
product_df.cache()

print(f"✅ Created {product_df.count():,} products")
product_df.printSchema()
product_df.show(5, truncate=False)

# ============ 2. CREATE SALES TABLE ============

print("\n💵 Creating Sales Table with 100,000+ transactions...")

# Generate sales data
num_transactions = 100000
sales = []
start_date = datetime(2023, 1, 1)
end_date = datetime(2024, 12, 31)

# Get product IDs for reference
product_ids = [row.product_id for row in product_df.select('product_id').collect()]
num_products = len(product_ids)

# Customer IDs
customer_ids = [f"CUST{str(i).zfill(6)}" for i in range(1, 1001)]

# Cache sugar products for performance
sugar_products_list = [row.product_id for row in product_df.filter('is_sugar_product == True').select('product_id').collect()]

for i in range(num_transactions):
    # Random date
    date_range = (end_date - start_date).days
    random_days = random.randint(0, date_range)
    sale_date = start_date + timedelta(days=random_days)
    
    # Number of items in this transaction (1-15)
    items_in_transaction = random.randint(1, 15)
    
    # Generate items for this transaction
    for item in range(items_in_transaction):
        # Select random product with bias towards sugar products
        if random.random() < 0.3:  # 30% chance to pick sugar product
            if sugar_products_list:
                product_id = random.choice(sugar_products_list)
            else:
                product_id = random.choice(product_ids)
        else:
            product_id = random.choice(product_ids)
        
        # Get product info efficiently
        product_row = product_df.filter(f'product_id == {product_id}').collect()
        if product_row:
            price = product_row[0].price
            sugar_content = product_row[0].sugar_content_grams
            is_sugar = product_row[0].is_sugar_product
        else:
            price = py_round(random.uniform(1, 50), 2)
            sugar_content = random.randint(0, 100)
            is_sugar = False
        
        quantity = random.randint(1, 10)
        
        # Introduce some data quality issues
        # 1. Null values in some fields
        address = random.choice([None, f"{random.randint(1, 999)} Main St", f"{random.randint(1, 999)} Park Ave"])
        
        # 2. Negative quantities (corrupt)
        if random.random() < 0.02:  # 2% chance
            quantity = -random.randint(1, 5)
        
        # 3. Zero price (corrupt)
        if random.random() < 0.015:  # 1.5% chance
            price = 0.0
        
        # 4. Negative price (corrupt)
        if random.random() < 0.01:  # 1% chance
            price = -py_round(random.uniform(1, 50), 2)
        
        # 5. Sugar content mismatch
        sugar_content_actual = sugar_content
        if random.random() < 0.02:  # 2% chance
            if is_sugar:
                sugar_content_actual = random.randint(0, 30)  # Should be high but low value
            else:
                sugar_content_actual = random.randint(70, 100)  # Should be low but high value
        
        # 6. Invalid dates (future)
        sale_date_actual = sale_date
        if random.random() < 0.01:
            sale_date_actual = datetime(2025, 1, 1) + timedelta(days=random.randint(1, 365))  # Future
        
        # 7. Missing product IDs
        product_id_actual = product_id
        if random.random() < 0.005:  # 0.5% chance
            product_id_actual = None
        
        # 8. Invalid quantities (decimal instead of integer)
        quantity_actual = quantity
        if random.random() < 0.01:
            quantity_actual = py_round(random.uniform(1.5, 10.5), 1)
        
        # Calculate total amount safely
        try:
            if isinstance(quantity_actual, (int, float)) and isinstance(price, (int, float)):
                total_amount = py_round(quantity_actual * price, 2)
            else:
                total_amount = 0.0
        except:
            total_amount = 0.0
        
        sales.append({
            'transaction_id': f"TRX{str(i + 1).zfill(8)}",
            'product_id': product_id_actual,
            'customer_id': random.choice(customer_ids),
            'sale_date': sale_date_actual,
            'quantity': quantity_actual,
            'unit_price': price,
            'total_amount': total_amount,
            'payment_method': random.choice(['Cash', 'Credit Card', 'Debit Card', 'UPI', 'PayPal']),
            'store_location': random.choice(['Store A - NYC', 'Store B - LA', 'Store C - Chicago', 
                                            'Store D - Houston', 'Store E - Phoenix', 'Store F - Philadelphia',
                                            'Store G - Dallas', 'Store H - Miami']),
            'customer_address': address,
            'sugar_content': sugar_content_actual if isinstance(sugar_content_actual, int) else 0,
            'discount_applied': random.choice([0, 5, 10, 15, 20, 25]) if random.random() < 0.3 else 0,
            'is_corrupt': False  # Will be flagged later
        })

# Create sales DataFrame
sales_df = spark.createDataFrame(sales)
sales_df.cache()

print(f"✅ Created {sales_df.count():,} sales records")
sales_df.printSchema()
sales_df.show(5, truncate=False)

# ============ 3. SAVE DATA ============

print("\n💾 Saving Data...")

# Create directories
os.makedirs("Data/grocery_data", exist_ok=True)

# Save product data
product_df.write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv("Data/grocery_data/products.csv")

print("✅ Products saved to: Data/grocery_data/products.csv")

# Save sales data
sales_df.write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv("Data/grocery_data/sales.csv")

print("✅ Sales saved to: Data/grocery_data/sales.csv")

# ============ 4. DATA QUALITY SUMMARY ============

from pyspark.sql.functions import col, current_date, min, max, count, sum, avg

print("\n" + "="*60)
print("📊 GROCERY DATA QUALITY SUMMARY")
print("="*60)

# Product statistics
total_products = product_df.count()
sugar_products = product_df.filter(col('is_sugar_product') == True).count()
non_sugar_products = product_df.filter(col('is_sugar_product') == False).count()
total_categories = product_df.select('category').distinct().count()

print(f"\n📦 Products:")
print(f"  - Total products: {total_products:,}")
print(f"  - Sugar products: {sugar_products:,}")
print(f"  - Non-sugar products: {non_sugar_products:,}")
print(f"  - Categories: {total_categories}")

# Sales statistics
total_transactions = sales_df.count()
unique_customers = sales_df.select('customer_id').distinct().count()

# Get date range using selectExpr
date_stats = sales_df.selectExpr(
    "min(sale_date) as min_date",
    "max(sale_date) as max_date"
).collect()[0]

print(f"\n💵 Sales:")
print(f"  - Total transactions: {total_transactions:,}")
print(f"  - Unique customers: {unique_customers:,}")
print(f"  - Date range: {date_stats['min_date']} to {date_stats['max_date']}")

# Data quality issues
negative_qty = sales_df.filter(col('quantity') < 0).count()
price_issues = sales_df.filter(col('unit_price') <= 0).count()
null_product = sales_df.filter(col('product_id').isNull()).count()
future_dates = sales_df.filter(col('sale_date') > current_date()).count()

print(f"\n❌ Data Quality Issues:")
print(f"  - Negative quantities: {negative_qty:,}")
print(f"  - Zero/negative prices: {price_issues:,}")
print(f"  - Null product IDs: {null_product:,}")
print(f"  - Future dates: {future_dates:,}")

print("\n📁 All files saved in: Data/grocery_data/")
print("  - products.csv")
print("  - sales.csv")
print("="*60)

# Unpersist to free memory
product_df.unpersist()
sales_df.unpersist()

print("\n✅ Grocery data creation complete!")

📦 Creating Product Table with 5000+ products...


✅ Created 200 products
root
 |-- brand: string (nullable = true)
 |-- category: string (nullable = true)
 |-- cost: double (nullable = true)
 |-- expiry_days: long (nullable = true)
 |-- is_sugar_product: boolean (nullable = true)
 |-- price: double (nullable = true)
 |-- product_id: long (nullable = true)
 |-- product_name: string (nullable = true)
 |-- reorder_level: long (nullable = true)
 |-- stock_quantity: long (nullable = true)
 |-- sub_category: string (nullable = true)
 |-- sugar_content_grams: long (nullable = true)
 |-- supplier: string (nullable = true)
 |-- weight_kg: double (nullable = true)

+-------------+--------------+-----+-----------+----------------+-----+----------+---------------+-------------+--------------+------------+-------------------+----------+---------+
|brand        |category      |cost |expiry_days|is_sugar_product|price|product_id|product_name   |reorder_level|stock_quantity|sub_category|sugar_content_grams|supplier  |weight_kg|
+-------------+-------

In [ ]:

# ============ 3. IDENTIFY DATA QUALITY ISSUES ============

print("\n🔍 Checking Data Quality Issues...")

# 1. Check for NULL values
print("\n📊 NULL Value Analysis:")
for col_name in sales_df.columns:
    null_count = sales_df.filter(col(col_name).isNull()).count()
    if null_count > 0:
        print(f"  - {col_name}: {null_count} null values ({null_count/sales_df.count()*100:.2f}%)")

# 2. Check for negative quantities
negative_quantity = sales_df.filter(col('quantity') < 0)
print(f"\n📊 Negative Quantities: {negative_quantity.count()}")

# 3. Check for zero/negative prices
price_issues = sales_df.filter(col('unit_price') <= 0)
print(f"📊 Zero/Negative Prices: {price_issues.count()}")

# 4. Check sugar content consistency
sugar_inconsistent = sales_df.filter(
    (col('sugar_content') < 30) & (col('product_id').isNotNull())
)
# Join with product table to check sugar products
sugar_products_ids = [row.product_id for row in product_df.filter('is_sugar_product == True').select('product_id').collect()]
if sugar_products_ids:
    sugar_products_df = sales_df.filter(col('product_id').isin(sugar_products_ids))
    low_sugar_in_sugar_products = sugar_products_df.filter(col('sugar_content') < 30)
    print(f"📊 Sugar products with low sugar content: {low_sugar_in_sugar_products.count()}")
    
    # Non-sugar products with high sugar content
    non_sugar_products_ids = [row.product_id for row in product_df.filter('is_sugar_product == False').select('product_id').collect()]
    if non_sugar_products_ids:
        non_sugar_df = sales_df.filter(col('product_id').isin(non_sugar_products_ids))
        high_sugar_in_non_sugar = non_sugar_df.filter(col('sugar_content') > 70)
        print(f"📊 Non-sugar products with high sugar content: {high_sugar_in_non_sugar.count()}")

# 5. Check for future dates
future_dates = sales_df.filter(col('sale_date') > current_date())
print(f"📊 Future Dates: {future_dates.count()}")

# 6. Check for duplicate transactions
duplicates = sales_df.groupBy('transaction_id').count().filter(col('count') > 1)
print(f"📊 Duplicate Transactions: {duplicates.count()}")

# 7. Check for invalid quantities (floating instead of integer)
from pyspark.sql.functions import col, cast
invalid_quantity = sales_df.filter(cast(col('quantity'), 'int') != col('quantity'))
print(f"📊 Invalid Quantities (decimal): {invalid_quantity.count()}")

# ============ 4. SAVE DATA WITH QUALITY ISSUES ============

print("\n💾 Saving Data...")

# Save product data
os.makedirs("Data/grocery_data", exist_ok=True)

product_df.coalesce(1).write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv("Data/grocery_data/products.csv")

print("✅ Products saved to: Data/grocery_data/products.csv")

# Save sales data with corruption markers
sales_with_issues = sales_df.withColumn(
    'quality_issue_flag',
    when(
        (col('quantity') < 0) |
        (col('unit_price') <= 0) |
        (col('sale_date') > current_date()) |
        (col('product_id').isNull()) |
        (cast(col('quantity'), 'int') != col('quantity')),
        'YES'
    ).otherwise('NO')
)

sales_with_issues.coalesce(1).write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv("Data/grocery_data/sales.csv")

print("✅ Sales saved to: Data/grocery_data/sales.csv")

# ============ 5. DATA QUALITY SUMMARY ============

print("\n" + "="*60)
print("📊 GROCERY DATA QUALITY SUMMARY")
print("="*60)

print(f"\n📦 Products:")
print(f"  - Total products: {product_df.count():,}")
print(f"  - Sugar products: {product_df.filter('is_sugar_product == True').count():,}")
print(f"  - Non-sugar products: {product_df.filter('is_sugar_product == False').count():,}")
print(f"  - Categories: {product_df.select('category').distinct().count()}")

print(f"\n💵 Sales:")
print(f"  - Total transactions: {sales_df.count():,}")
print(f"  - Unique customers: {sales_df.select('customer_id').distinct().count():,}")
print(f"  - Date range: {sales_df.agg({'sale_date': 'min'}).collect()[0][0]} to {sales_df.agg({'sale_date': 'max'}).collect()[0][0]}")

print(f"\n❌ Data Quality Issues:")
print(f"  - Negative quantities: {sales_df.filter(col('quantity') < 0).count():,}")
print(f"  - Zero/negative prices: {sales_df.filter(col('unit_price') <= 0).count():,}")
print(f"  - Null product IDs: {sales_df.filter(col('product_id').isNull()).count():,}")
print(f"  - Future dates: {sales_df.filter(col('sale_date') > current_date()).count():,}")
print(f"  - Sugar content inconsistencies: {len(sugar_inconsistent.collect()):,}")

print("\n📁 All files saved in: Data/grocery_data/")
print("  - products.csv")
print("  - sales.csv")
print("="*60)

# ============ 6. SUGAR PRODUCT ANALYSIS ============

print("\n🍬 Sugar Product Analysis:")
print("-"*30)

# Top sugar products by sales
sugar_sales = sales_df.join(product_df, 'product_id').filter('is_sugar_product == True')
top_sugar_products = sugar_sales.groupBy('product_name', 'sub_category') \
    .agg(
        sum('quantity').alias('total_quantity'),
        sum('total_amount').alias('total_revenue'),
        avg('unit_price').alias('avg_price'),
        avg('sugar_content').alias('avg_sugar_content')
    ) \
    .orderBy(col('total_revenue').desc()) \
    .limit(10)

print("🏆 Top 10 Sugar Products by Revenue:")
top_sugar_products.show(truncate=False)

# ============ 7. SAMPLE QUERIES FOR AQE TESTING ============

print("\n📊 Sample Queries for AQE Testing:")
print("-"*30)

# Query 1: Sugar sales by category
print("\n1. Sugar Sales by Category:")
sales_with_products = sales_df.join(product_df, 'product_id')
sugar_category_sales = sales_with_products.filter('is_sugar_product == True') \
    .groupBy('category') \
    .agg(
        sum('quantity').alias('total_quantity'),
        sum('total_amount').alias('total_revenue'),
        avg('sugar_content').alias('avg_sugar_content')
    ) \
    .orderBy(col('total_revenue').desc())

sugar_category_sales.show(truncate=False)

# Query 2: Sugar content anomalies
print("\n2. Sugar Content Anomalies:")
sugar_anomalies = sales_with_products.filter(
    ((col('is_sugar_product') == True) & (col('sugar_content') < 20)) |
    ((col('is_sugar_product') == False) & (col('sugar_content') > 70))
).select('product_name', 'category', 'sugar_content', 'is_sugar_product', 'transaction_id')
sugar_anomalies.show(10, truncate=False)

print(f"\n✅ Total anomalies: {sugar_anomalies.count()}")

# Unpersist to free memory
product_df.unpersist()
sales_df.unpersist()

print("\n✅ Grocery data creation complete!")
print("📊 Ready for AQE (Automatic Query Execution) testing!")